In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from multimodal_lancedb import MusicDatabase
from utils import (
    EmbeddingProcessor,
    Ranker,
    MusicSearchSystem
)
from IPython.display import Audio, display
import pandas as pd

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
search_system.import_text_descriptions("final_dataset.csv", process_audio=True)

Successfully imported 200 records from final_dataset.csv
Processed audio for: My Rhapsody Sounds - Short Version A
Processed audio for: Laid Back - Short Version A
Processed audio for: Far Taj
Processed audio for: The Stones - Short Version
Processed audio for: Fixed - Short Version B
Processed audio for: Inside the War
Processed audio for: Locked in Silence - Short Version C
Processed audio for: How Could I Know - Intro
Processed audio for: Coming Over - Short Version B
Processed audio for: The 8 Oclock Story - Intro
Processed audio for: Modern Life Is Thin and Shallow
Processed audio for: Magnetic Storms - Short Version
Processed audio for: Preparing the Cannons
Processed audio for: Sharp Knives - Instrumental Version
Processed audio for: Rundown
Processed audio for: Be Quiet
Processed audio for: Little One - Short Version
Processed audio for: Begin Again - Short Version B
Processed audio for: Return - Short Version
Processed audio for: Energetic Loop
Processed audio for: The Most Im

In [4]:
import lancedb
db = lancedb.connect("./.lancedb_2")
table_audio = db.open_table("music_audio")
audio_embedding_df = table_audio.to_pandas()

audio_embedding_df

,song_name,song_path,audio_vector
0,My Rhapsody Sounds - Short Version A,music/Assaf Ayalon - My Rhapsody Sounds - Shor...,"[-0.029478367, 0.009709472, 0.05185532, 0.0254..."
1,Laid Back - Short Version A,music/The Mind Sweepers - Laid Back - Short Ve...,"[-0.033787563, 0.040226605, 0.004442984, 0.081..."
2,Far Taj,music/ZISO - Far Taj.mp3,"[-0.018721217, 0.035051364, 0.05141652, 0.0356..."
3,The Stones - Short Version,music/Wild Tulip - The Stones - Short Version.mp3,"[-0.008583562, 0.020294761, 0.0050604204, 0.03..."
4,Fixed - Short Version B,music/Swirling Ship - Fixed - Short Version B.mp3,"[0.0031058518, 0.020443501, 0.0060475464, -0.0..."
...,...,...,...
195,Orchestral News Intro,music/Tomasz_Redman - Orchestral News Intro.mp3,"[0.010141604, -0.017981885, 0.014272054, -0.03..."
196,Upbeat Happy Fun Logo,music/puremusic - Upbeat Happy Fun Logo.mp3,"[-0.031311695, -0.03104818, 0.022266045, 0.022..."
197,Happy Birthday In Paris,music/Music-Ideas - Happy Birthday In Paris.mp3,"[-0.049790498, -0.03626891, 0.059174698, -0.00..."
198,Funny Game Loop,music/honey_lemon - Funny Game Loop.wav,"[-0.045391183, -0.033096816, 0.025908915, -0.0..."


In [5]:
table_text = db.open_table("music_text")
text_embedding_df = table_text.to_pandas()

text_embedding_df

,source,song_name,artist,mood,video_theme,genre,instrument,other_tags,bpm,lmm_description,combined_info,text_vector
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Cinematic, Acoustic, Pop, Folk, Children, Corp...","Acoustic Guitar, Keys",,145.0,A positive and uplifting acoustic folk track w...,"Moods: Uplifting, Happy, Carefree, Love, Playf...","[0.00016941165, -0.011131651, -0.004014101, -0..."
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry",Rock,"Electric, Guitar, Acoustic Drums",,78.0,This is a powerful and energetic rock music tr...,"Moods: Powerful, Serious, Angry. Video Themes:...","[-0.0041819224, -0.01964485, -0.021090291, -0...."
2,Artlist,Far Taj,ZISO,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","World, Electronic, Hip Hop","Ethnic, Electronic Drums, Bass",,96.0,A traditional Indian Bhangra track with modern...,"Moods: Uplifting, Powerful, Carefree, Groovy. ...","[-0.011723319, -0.008885955, 0.0040757894, -0...."
3,Artlist,The Stones - Short Version,Wild Tulip,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Cinematic,Piano,,69.0,This piece is a solo piano instrumental with a...,"Moods: Love, Serious, Dramatic, Sad, Hopeful. ...","[0.0012076573, -0.0034555339, -0.0020917628, -..."
4,Artlist,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,121.0,"The music is mysterious and dramatic, featurin...","Moods: Serious, Dramatic, Scary, Dark. Video T...","[-0.0021377725, -0.012590199, -0.013713269, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",corporate,strings,global,125.0,This is a dynamic and uplifting music track th...,"Moods: energetic, epic, powerful, solemn, upli...","[-0.006809455, -0.016746698, -0.016968248, -0...."
196,envato,Upbeat Happy Fun Logo,puremusic,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","acoustic, children","claps, ukulele","melody, youth",NaN,"A positive, upbeat, cheerful, and happy acoust...","Moods: bouncy, bright, catchy, cheerful, energ...","[0.002132031, -0.005020746, -0.0029374287, -0...."
197,envato,Happy Birthday In Paris,Music-Ideas,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","bigband, jazz, retro","accordion, piano, trumpets","france, french, paris",120.0,A fun and lively Latin track featuring a varie...,"Moods: cheerful, funny, happy, lively, playful...","[-0.012678004, -0.011919039, -0.00089095806, -..."
198,envato,Funny Game Loop,honey_lemon,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv","acoustic, children, folk, jazz",bells,loop,170.0,"A casual, jazzy, swing music with vibraphone, ...","Moods: comical, fun, funny, laugh, smile, soft...","[-0.012116052, -0.01674075, 0.010771282, -0.02..."


In [7]:
text_embedding_df['combined_info']

0      Moods: Uplifting, Happy, Carefree, Love, Playf...
1      Moods: Powerful, Serious, Angry. Video Themes:...
2      Moods: Uplifting, Powerful, Carefree, Groovy. ...
3      Moods: Love, Serious, Dramatic, Sad, Hopeful. ...
4      Moods: Serious, Dramatic, Scary, Dark. Video T...
                             ...                        
195    Moods: energetic, epic, powerful, solemn, upli...
196    Moods: bouncy, bright, catchy, cheerful, energ...
197    Moods: cheerful, funny, happy, lively, playful...
198    Moods: comical, fun, funny, laugh, smile, soft...
199    Moods: beautiful, calm, dramatic, dreamy, eleg...
Name: combined_info, Length: 200, dtype: object